# peak5 — Phase 2 inference (public LB **0.89552**, #1)

Reproduces the current leaderboard entry. The shipped model is a **uniform weight average of
five checkpoints from a single training run** — no ensembling, no extra inference cost.

| | |
|---|---|
| dev[0:300] | Token F1 **0.8348** · ROUGE-L **0.8061** |
| dev[300:600] *(disjoint, nothing selected on it)* | Token F1 **0.8404** |
| public LB | **0.89552** |
| params | **247,577,856** — one model, within the 3B cap |

## How the shipped weights were produced

The base run is `E05/english_draft`: BanglaT5 fine-tuned for register transfer on
`english + draft` input, 768/512, effective batch 64, lr 1e-3, adafactor, seed 11, bf16,
budget 30,000 steps with early stopping (patience 8). It peaked at **step 12,000** and
stopped at 14,000. That single best checkpoint scores **0.8328** and is what earned 0.89347.

`peak5` is the equal-weight mean of the five checkpoints **surrounding** that peak:

    checkpoint-11500, 11750, 12000, 12250, 12500   ->  mean of state_dicts  ->  peak5

    sd = {k: sum(m[k].float() for m in members) / len(members) for k in members[0]}

Two things about this that are not the usual recipe:

* It is **peak-centred, not tail-centred.** Averaging the *last* N checkpoints — the common
  formulation — is worse here: tail3 0.8330, tail5 0.8337, tail7 0.8323, all below peak5's
  0.8348. The window has to straddle the maximum, which means the run must be trained *past*
  its peak to build it.
* **It does not compose with seed averaging.** Averaging two same-schedule seeds gives 0.8345,
  and combining that with peak5 gives 0.8333 — worse than either alone. Both methods appear to
  remove the same variance, so 0.8348 is a ceiling reached by two independent routes, not a
  gain that stacks.

`checkpoint_average.json` in the attached dataset records the exact member list and the
score measured at averaging time.

🔴 The last cell diffs this notebook's output against the exact CSV that scored 0.89552.
That diff — not a weight hash — is the reproduction evidence: floating-point paths differ
between an H100 in bf16 and a T4 in fp32, so hashes will not match and *outputs* must.

In [ ]:
# 1 — pinned libs. transformers 5.x breaks T5 training/inference (trap #00).
!pip install -q --upgrade "transformers==4.57.3" git+https://github.com/csebuetnlp/normalizer
import transformers
assert transformers.__version__ == "4.57.3", transformers.__version__
from normalizer import normalize
print("transformers", transformers.__version__, "| normalizer OK:", normalize("হেলো,  নাসেনিয়া ডকে"))

In [ ]:
# 2 — hardware gate. Kaggle PyTorch ships sm_70+ only; a P100 (sm_60) cannot run
# anything, and T4 (sm_75) has NO bf16 hardware, so this must be fp32.
import torch, glob, os
assert torch.cuda.is_available(), "no GPU"
cap = torch.cuda.get_device_capability()
assert cap[0] >= 7, f"sm_{cap[0]}{cap[1]} unsupported — need T4 (sm_75)"
# 🔴 gate on capability, NOT is_bf16_supported(): that returns True on a T4 via emulation,
# which is slower than fp32 and not what the model was validated in.
BF16 = cap[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float32
print(f"{torch.cuda.get_device_name(0)} sm_{cap[0]}{cap[1]} -> {DTYPE}")

CKPT = os.path.dirname(glob.glob("/kaggle/input/**/best/config.json", recursive=True)[0])
TEST = glob.glob("/kaggle/input/**/test_english_draft.parquet", recursive=True)[0]
DEV  = glob.glob("/kaggle/input/**/dev_english_draft.parquet", recursive=True)[0]
REF  = glob.glob("/kaggle/input/**/submission_reference.csv", recursive=True)[0]
AVG  = glob.glob("/kaggle/input/**/checkpoint_average.json", recursive=True)[0]
print("ckpt:", CKPT)

In [ ]:
# 3 — provenance of the averaged weights, read straight off the artefact.
import json
avg = json.load(open(AVG))
print("members averaged (equal weight):")
for m in avg["members"]:
    print("   ", m.split("/")[-1])
print(f"\nscore at averaging time : {avg['dev_token_f1']:.6f}")
print(f"parameters              : {avg['params']:,}")
assert len(avg["members"]) == 5, avg["members"]
assert avg["params"] <= 3_000_000_000
# the average of N same-shaped models has the SAME parameter count as one of them —
# this is why it costs nothing at inference, unlike an N-member ensemble.

In [ ]:
# 4 — load, verify the parameter cap and that the logits are finite.
# 🔴 T5 overflows to NaN in fp16 SILENTLY: it still prints a parameter count, still
# "decodes", and still writes a well-formed CSV of garbage. Probe before trusting it.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForSeq2SeqLM.from_pretrained(CKPT, dtype=DTYPE).cuda().eval()

n = sum(p.numel() for p in model.parameters())
print(f"parameters: {n:,}")
assert n <= 3_000_000_000, f"3B CAP BREACHED: {n:,}"
assert n == avg["params"], f"loaded {n:,} != recorded {avg['params']:,}"
print(f"✅ within the 3B cap (headroom {(3_000_000_000-n)/1e6:.0f}M)")

with torch.no_grad():
    probe = tok("হেলো", return_tensors="pt").to("cuda")
    lg = model(**probe, decoder_input_ids=torch.zeros((1,1), dtype=torch.long, device="cuda")).logits
assert torch.isfinite(lg).all(), "❌ NON-FINITE LOGITS — wrong dtype"
print("✅ logits finite")

In [ ]:
# 5 — the decoder. Unchanged from the 0.89347 entry: averaging the weights did not move
# the decoder optimum. A fresh sweep over beams {4,8,12,16} x length_penalty {1.0 … 3.2}
# re-picked these same settings at Δ +0.0000, and max_new_tokens is inert (320/400/512
# score bit-identically — nothing this model emits exceeds ~100 tokens).
import pandas as pd
GEN = dict(num_beams=8, min_new_tokens=0, max_new_tokens=320,
           length_penalty=1.2, do_sample=False)
# max_source_len 768 = the cap it TRAINED at. Decoding at a smaller cap silently feeds the
# model a fraction of its input and quietly costs score.
MAX_SRC, BATCH = 768, 16

@torch.no_grad()
def generate(texts):
    out = []
    for i in range(0, len(texts), BATCH):
        enc = tok([normalize(str(t)) for t in texts[i:i+BATCH]], return_tensors="pt",
                  padding=True, truncation=True, max_length=MAX_SRC).to("cuda")
        g = model.generate(**enc, **GEN)
        out += tok.batch_decode(g, skip_special_tokens=True)
        if i % (BATCH*10) == 0: print(f"  {i}/{len(texts)}", flush=True)
    return out

In [ ]:
# 6 — 🔴 KNOWN-GOOD GATE. Score the 300 dev rows the checkpoint window was selected on.
# "Assert a recorded dev number before writing anything" — the only thing that ever caught
# the fp16 bug. A well-formed CSV of garbage is indistinguishable from a good one until
# the leaderboard says so.
import re
from collections import Counter

def tokenize(s):
    return re.findall(r"[ঀ-৿]+|[A-Za-z]+|\d+", str(s))

def token_f1(p, r):
    pt, rt = Counter(tokenize(p)), Counter(tokenize(r))
    ov = sum((pt & rt).values())
    if not ov: return 0.0
    prec, rec = ov/max(1,sum(pt.values())), ov/max(1,sum(rt.values()))
    return 2*prec*rec/(prec+rec)

devall = pd.read_parquet(DEV)
dev = devall.iloc[:300]
dp = generate(dev["input"].tolist())
f1 = sum(token_f1(a, b) for a, b in zip(dp, dev["output"])) / len(dev)
print(f"\ndev[0:300] Token F1 = {f1:.4f}   (recorded: 0.8348)")
assert f1 > 0.82, f"❌ {f1:.4f} is far below the recorded 0.8348 — do NOT submit this"
print("✅ matches the recorded average")

In [ ]:
# 7 — the disjoint check, re-run here rather than just quoted. Rows [300:600] were never
# used to pick the checkpoint window, the decoder, or anything else. peak5 scores HIGHER
# there than on the selection rows, which is the opposite of what a spurious winner does.
devB = devall.iloc[300:600]
bp = generate(devB["input"].tolist())
f1b = sum(token_f1(a, b) for a, b in zip(bp, devB["output"])) / len(devB)
print(f"\ndev[300:600] Token F1 = {f1b:.4f}   (recorded: 0.8404)")
print(f"single best checkpoint scored 0.8348 on these same rows -> peak5 gains {f1b-0.8348:+.4f}")

In [ ]:
# 8 — test split -> submission.csv
test = pd.read_parquet(TEST)
preds = generate(test["input"].tolist())
sub = pd.DataFrame({"id": test["id"], "output": preds})
assert len(sub) == 1000 and list(sub.columns) == ["id", "output"]
assert not sub["id"].duplicated().any() and not (sub["output"].str.strip() == "").any()
sub.to_csv("submission.csv", index=False)
print(sub.shape, list(sub.columns))
sub.head(3)

In [ ]:
# 9 — 🔴 PHASE 2 EVIDENCE: diff against the exact CSV that scored 0.89552.
ref = pd.read_csv(REF)
m = ref.merge(sub, on="id", suffixes=("_ref", "_new"))
assert len(m) == 1000, "id mismatch against the scored submission"
exact = (m["output_ref"].str.strip() == m["output_new"].str.strip()).sum()
agree = sum(token_f1(a, b) for a, b in zip(m["output_ref"], m["output_new"])) / len(m)
print(f"identical rows : {exact}/1000")
print(f"mean Token F1 vs the scored submission : {agree:.4f}")
if exact == 1000:
    print("✅ EXACT reproduction of the 0.89552 leaderboard entry")
else:
    print("⚠️  not byte-identical — expected when the scored CSV was produced in bf16 on an\n"
          "    A800 and this runs fp32 on a T4. Token F1 above 0.99 means the same model,\n"
          "    same decoder, different floating-point path.")
    assert agree > 0.99, f"❌ only {agree:.4f} agreement — this is NOT the same model"